In [1]:
!pip install -r requirements.txt
from warpdrive import WarpDrive


wd = WarpDrive()

data = wd.get_args("data")
model = wd.get_args("model")
# your code here


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 159.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 246.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.7/166.7 MB 184.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: psutil
    Found existing installation: psutil 7.0.0
    Uninstalling psutil-7.0.0:
      Successfully uninstalled psutil-7.0.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.2
    Uninstalling numpy-1.24.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [numpy]
      Successfully uninstalled numpy-1.24.2━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [numpy]
  Attempting uninstall: scipym━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [numpy]
    Found existing installation: scipy 1.15.2━━━━━━━━━━━━━━━━━ 1/5 [numpy]
    Uninstalling scipy-1.15.2:90m╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [scipy]
      Successfully uninstalled scipy-1.15.2━━━━━━━━━━━━━━━━━━━━━━━ 2/5 

<IPython.core.display.Javascript object>

Directory '../models/' does not exist.


/usr/local/lib/python3.10/site-packages/xgboost/compat.py:31: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


Model loaded successfully


In [2]:
# Import necessary library
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

In [3]:
X_test= data[['fico', 'cnt_units', 'dti','cltv', 'ltv',
              'int_rt', 'orig_loan_term', 'occpy_sts_lable_encoded',
              'channel_lable_encoded','prop_type_lable_encoded','loan_purpose_lable_encoded']]

In [4]:
y_test=data['def_flag']

In [5]:
# Get predicted probabilities
y_proba = model.predict_proba(X_test)

# y_proba[:, 1] gives the probability of the positive class (class 1)
positive_class_proba = y_proba[:, 1]

# Optional: print or inspect the probabilities
#print(positive_class_proba)

In [6]:
positive_class_proba_series = pd.Series(positive_class_proba,
                                        index=X_test.index, name="predicted_probability")
#positive_class_proba_series

In [7]:
# Convert to Series
y_actual_series = pd.Series(y_test, name="actual")

# Optional: keep index aligned with X_test
y_actual_series.index = X_test.index

In [8]:
def calculate_concordance_metrics(y_actual: pd.Series, predictions: pd.Series):
    # Convert pandas Series to numpy arrays
    y_actual_np = y_actual.to_numpy()
    predictions_np = predictions.to_numpy()

    # Initialize counters
    concordant = 0
    discordant = 0
    tied = 0

    # Calculate total number of pairs
    n = len(y_actual_np)
    total_pairs = 0

    # Calculate concordant, discordant, and tied pairs
    for i in range(n):
        for j in range(i + 1, n):
            if y_actual_np[i] != y_actual_np[j]:  # Different actual values
                total_pairs += 1
                if (predictions_np[i] > predictions_np[j] and y_actual_np[i] > y_actual_np[j]) or \
                   (predictions_np[i] < predictions_np[j] and y_actual_np[i] < y_actual_np[j]):
                    concordant += 1
                elif (predictions_np[i] > predictions_np[j] and y_actual_np[i] < y_actual_np[j]) or \
                     (predictions_np[i] < predictions_np[j] and y_actual_np[i] > y_actual_np[j]):
                    discordant += 1
            else:  # Same actual values
                if predictions_np[i] == predictions_np[j]:  # Predictions are tied
                    tied += 1

    # Compute Percent Concordant, Discordant, and Tied
    percent_concordant = (concordant / total_pairs) * 100
    percent_discordant = (discordant / total_pairs) * 100
    percent_tied = (tied / total_pairs) * 100

    # Calculate Somers' D
    somers_d = (concordant - discordant) / total_pairs

    # Calculate Gamma
    gamma = (concordant - discordant) / (concordant + discordant)

    # Calculate Tau-a
    tau_a = 2 * (concordant - discordant) / (n * (n - 1))

    # Calculate c statistic (AUC)
    c_stat = percent_concordant / 100 + (percent_tied / 200)

    # Print results
    # print("Percent Concordant:", round(percent_concordant, 1))
    # print("Percent Discordant:", round(percent_discordant, 1))
    # print("Percent Tied:", round(percent_tied, 1))
    # print("Total Pairs:", total_pairs)
    # print("Somers' D:", round(somers_d, 3))
    # print("Gamma:", round(gamma, 3))
    # print("Tau-a:", round(tau_a, 3))
    # print("c:", round(c_stat, 3))

    # Return results as a dictionary for further use
    return pd.DataFrame([{
        "percent_concordant": percent_concordant,
        "percent_discordant": percent_discordant,
        "percent_tied": percent_tied,
        "total_pairs": total_pairs,
        "somers_d": somers_d,
        "gamma": gamma,
        "tau_a": tau_a,
        "c_stat": c_stat
    }])

In [9]:
output=calculate_concordance_metrics(y_test, positive_class_proba_series)
wd.save_table(output,"Concordant Discordant Table")

Dataframe Concordant Discordant Table will be shown as output in jobs.


,percent_concordant,percent_discordant,percent_tied,total_pairs,somers_d,gamma,tau_a,c_stat
0,86.24383,13.756033,0.032377,7289100,0.724878,0.724879,0.362461,0.8626
